# Sample ARC Submission

This is a sample notebook that can help get you started with creating an ARC Prize submission. It covers the basics of loading libraries, loading data, implementing an approach, and submitting.

You should be able to submit this notebook to the evaluation portal and have it run successfully (although you'll get a score of 0, so you'll need to do some work if you want to do better!)


# Load needed libraries

Basic libraries like numpy, torch, matplotlib, and tqdm are already installed.

In [8]:
import json
from tqdm import tqdm

# Load the data

Here we are loading the training challenges and solutions (this is the public training set), the evaluation challenges and solutions (this is the public evaluation set), and the test challenges (currently a placeholder file that is a copy of the public evaluation challanges).

For your initial testing and exploration, I'd recommend not using the public evaluation set, just work off the public training set and then test against the test challenges (which is actually the public evaluation set). However, when competing in the competition, then you can should probably use the evaluation tasks for training too.

In [9]:
# Public training set
train_challenges_path = '../input/arc-prize-2024/arc-agi_training_challenges.json'
train_solutions_path = '../input/arc-prize-2024/arc-agi_training_solutions.json'

with open(train_challenges_path) as fp:
    train_challenges = json.load(fp)
with open(train_solutions_path) as fp:
    train_solutions = json.load(fp)

# Public evaluation set
evaluation_challenges_path = '../input/arc-prize-2024/arc-agi_evaluation_challenges.json'
evaluation_solutions_path = '../input/arc-prize-2024/arc-agi_evaluation_solutions.json'

with open(evaluation_challenges_path) as fp:
    evaluation_challenges = json.load(fp)
with open(evaluation_solutions_path) as fp:
    evaluation_solutions = json.load(fp)

# This will be the hidden test challenges (currently has a placeholder to the evaluation set)
test_challenges_path = '../input/arc-prize-2024/arc-agi_test_challenges.json'

with open(test_challenges_path) as fp:
    test_challenges = json.load(fp)

Here is an example of what a test task looks like:

In [10]:
sample_task = list(test_challenges.keys())[0]
test_challenges[sample_task]

{'test': [{'input': [[3, 2], [7, 8]]}],
 'train': [{'input': [[8, 6], [6, 4]],
   'output': [[8, 6, 8, 6, 8, 6],
    [6, 4, 6, 4, 6, 4],
    [6, 8, 6, 8, 6, 8],
    [4, 6, 4, 6, 4, 6],
    [8, 6, 8, 6, 8, 6],
    [6, 4, 6, 4, 6, 4]]},
  {'input': [[7, 9], [4, 3]],
   'output': [[7, 9, 7, 9, 7, 9],
    [4, 3, 4, 3, 4, 3],
    [9, 7, 9, 7, 9, 7],
    [3, 4, 3, 4, 3, 4],
    [7, 9, 7, 9, 7, 9],
    [4, 3, 4, 3, 4, 3]]}]}

# Generating a submission

To generate a submission you need to output a file called `submission.json` that has the following format:

```
{"00576224": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "009d5c81": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "12997ef3": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]},
              {"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 ...
}
```

In this case, the task ids come from `test_challenges`. There may be multiple (i.e., >1) test items per task. Therefore, the dictionary has a list of dicts for each task. These submission dictionaries should appear in the same order as the test items from `test_challenges`. Additionally, you can provide two attempts for each test item. In fact, you **MUST** provide two attempts. If you only want to generate a single attempt, then just submit the same answer for both attempts (or submit an empty submission like the ones shown in the example snippit just above.

Here is how we might create a blank submission:

In [11]:
# Create an empty submission dict for output
submission = {}

# iterate over the test items and build up submission answers
count = 0
for key, task in tqdm(test_challenges.items()):

    # Here are the task's training inputs and outputs
    train_inputs = [item['input'] for item in task['train']]
    train_outputs = [item['output'] for item in task['train']]

    # Here we generate outputs for each test item.
    submission[key] = []

    # this is just a placeholder, but would be where you might generate your predictions.
    blank_prediction = [[0, 0], [0, 0]]
    submission[key] = [{'attempt_1': blank_prediction, 'attempt_2': blank_prediction} for item in task['test']]

# Here we write the submissions to file, so that they will get evaluated
with open('submission.json', 'w') as fp:
    json.dump(submission, fp)

100%|██████████| 400/400 [00:00<00:00, 498876.48it/s]


Here is what our submission for the test task above looks like:

In [12]:
submission[sample_task]

[{'attempt_1': [[0, 0], [0, 0]], 'attempt_2': [[0, 0], [0, 0]]}]

# ARC Submission Attempt

In [13]:
from random import choice
from py_search.base import Problem
from py_search.base import Node
from py_search.uninformed import breadth_first_search, depth_first_search
from py_search.informed import best_first_search, iterative_deepening_best_first_search, widening_beam_search
from py_search.optimization import local_beam_search


class ARCPuzzle:
    """
    An eight puzzle class that can be used to test different search algorithms.
    When first created the puzzle is in the solved state.
    """

    def __init__(self, state):
        self.state = tuple((tuple(row) for row in state))

    def __hash__(self):
        return hash(self.state)

    def __eq__(self, other):
        if isinstance(other, ARCPuzzle):
            return self.state == other.state
        return False

    def __ne__(self, other):
        return not self.__eq__(other)

    def __repr__(self):
        return str(self)

    def __str__(self):
        out = ""
        for row in self.state:
            out += str(row) + "\n"
        return out

    def get_matrix(self):
        return [list(row) for row in self.state]

    def copy(self):
        """
        Makes a deep copy of an ARCPuzzle object.
        """
        new = ARCPuzzle(self.state)
        return new

    def randomize(self, num_shuffles):
        """
        Randomizes an ARCPuzzle by executing a random action `num_suffles` times.
        """
        for i in range(num_shuffles):
            actions = [a for a in self.legalActions()]
            a = choice([a for a in self.legalActions()])
            # print(actions, a)
            self.executeAction(a)

        return self

    def resize(self, grid, height, width):
        """
        Resizes the grid to the given height and width, padding with zeros if needed.
        """
        if len(grid) > height:
            grid = grid[:height]
        elif len(grid) < height:
            grid = list(grid) + [tuple([0] * len(grid[0]))
                                 for _ in range(height - len(grid))]
        grid = [row[:width] + (0,) * (width - len(row))
                if len(row) < width else row[:width] for row in grid]

        return tuple(grid)

    def tophalf(self, grid):
        """
        Returns the top half of the grid.
        """
        return grid[:len(grid) // 2]

    def rotate(self, grid, degrees):
        """
        Rotates the grid by the given number of degrees (90, 180, or 270).
        """
        if degrees == 90:
            return tuple(zip(*grid[::-1]))
        elif degrees == 180:
            return tuple(row[::-1] for row in grid[::-1])
        elif degrees == 270:
            return tuple(zip(*grid))[::-1]

    def pattern_repeat(self, times_horizontal=3, times_vertical=3):
        """
        Repeats the current pattern horizontally and vertically.
        """
        if not self.state:
            return self.state

        # First repeat horizontally
        horizontal = tuple(
            row * times_horizontal
            for row in self.state
        )

        # Then repeat vertically
        vertical = horizontal * times_vertical

        return tuple(vertical)

    def alternate_pattern(self):
        """
        Creates an alternating pattern by flipping elements in a checkerboard-like fashion.
        """
        if not self.state or not self.state[0]:
            return self.state

        rows = len(self.state)
        cols = len(self.state[0])

        new_state = []
        for i in range(rows):
            new_row = []
            for j in range(cols):
                if (i // 2) % 2 == 0:  # For even pairs of rows
                    new_row.append(self.state[i][j])
                else:  # For odd pairs of rows
                    # Swap elements in consecutive pairs
                    if j % 2 == 0 and j + 1 < cols:
                        new_row.append(self.state[i][j + 1])
                    elif j % 2 == 1:
                        new_row.append(self.state[i][j - 1])
                    else:
                        new_row.append(self.state[i][j])
            new_state.append(tuple(new_row))

        return tuple(new_state)

    def hmirror(self, grid):
        """
        Mirrors the grid horizontally.
        """
        return grid[::-1]

    def vmirror(self, grid):
        """
        Mirrors the grid vertically.
        """
        return tuple(reversed(grid))

    def lshift(self, grid):
        return tuple([tuple([e for e in row if e != 0] + [0]*row.count(0))
                      for row in grid])

    def flood_fill(self, grid, x, y, target_color):
        """
        Performs a flood fill starting from (x, y), replacing the starting cell's color with the target color.
        """
        grid = [list(row) for row in grid]
        if x < 0 or x >= len(grid) or y < 0 or y >= len(grid[0]):
            return tuple(tuple(row) for row in grid)
        start_color = grid[x][y]
        if start_color == target_color:
            return tuple(tuple(row) for row in grid)

        stack = [(x, y)]

        while stack:
            cx, cy = stack.pop()
            grid[cx][cy] = target_color
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = cx + dx, cy + dy
                if 0 <= nx < len(grid) and 0 <= ny < len(grid[0]) and grid[nx][ny] == start_color:
                    stack.append((nx, ny))

        return tuple(tuple(row) for row in grid)

    def compress(self, grid):
        """
        Removes frontiers of the grid.
        """
        ri = [i for i, r in enumerate(grid) if len(set(r)) == 1]
        ci = [j for j, c in enumerate(zip(*grid)) if len(set(c)) == 1]
        return tuple([tuple([v for j, v in enumerate(r) if j not in ci])
                      for i, r in enumerate(grid) if i not in ri])

    def crop(self, grid, x1, y1, x2, y2):
        """
        Crops the grid to the specified section from (x1, y1) to (x2, y2).
        """
        x1, x2 = max(0, x1), min(len(grid) - 1, x2)
        y1, y2 = max(0, y1), min(len(grid[0]) - 1, y2)

        cropped_grid = tuple(tuple(row[y1:y2+1]) for row in grid[x1:x2+1])

        return cropped_grid

    def scale(self, grid, factor):
        """
        Scales the grid by the given factor.
        """
        if factor > 1:
            factor = int(factor)
            return tuple(
                tuple(grid[i // factor][j // factor]
                      for j in range(len(grid[0]) * factor))
                for i in range(len(grid) * factor)
            )
        elif factor < 1:
            factor = int(1 / factor)
            return tuple(
                tuple(grid[i * factor][j * factor]
                      for j in range(len(grid[0]) // factor))
                for i in range(len(grid) // factor)
            )
        else:
            return grid

    def mapcolor(self, grid, a, b):
        return tuple(tuple(b if e == a else e for e in row) for row in grid)

    def trim(self, grid):
        """ removes border """
        return tuple(r[1:-1] for r in grid[1:-1])

    def executeAction(self, action):
        """
        Executes an action to the ARCPuzzle object.

        :param action: the action to execute
        :type action: "up", "left", "right", or "down"
        """
        if action == 'tophalf':
            self.state = self.tophalf(self.state)
        elif action == 'hmirror':
            self.state = self.hmirror(self.state)
        elif action == 'vmirror':
            self.state = self.vmirror(self.state)
        elif action == 'lshift':
            self.state = self.lshift(self.state)
        elif action == 'compress':
            self.state = self.compress(self.state)
        elif action[:8] == 'mapcolor':
            args = action[9:-1].split(',')
            self.state = self.mapcolor(self.state, int(args[0]), int(args[1]))
        elif action[:10] == 'flood_fill':
            args = action[11:-1].split(',')
            x, y, target_color = map(int, args)
            self.state = self.flood_fill(self.state, x, y, target_color)
        elif action[:4] == 'crop':
            args = action[5:-1].split(',')
            x1, y1, x2, y2 = map(int, args)
            self.state = self.crop(self.state, x1, y1, x2, y2)
        elif action[:5] == 'scale':
            factor = float(action[6:-1])
            self.state = self.scale(self.state, factor)
        elif action[:6] == 'rotate':
            degrees = int(action[7:-1])
            self.state = self.rotate(self.state, degrees)
        elif action == 'pattern_repeat':
            self.state = self.pattern_repeat()
        elif action == 'alternate_pattern':
            self.state = self.alternate_pattern()

    def legalActions(self):
        """
        Returns an iterator to the legal actions that can be executed in the
        current state.
        """
        for action in ['tophalf', 'hmirror', 'vmirror', 'lshift', 'compress', 'alternate_pattern']:
            yield action

        for a in set(e for row in self.state for e in row):
            for b in range(10):
                if a == b:
                    continue
                yield f'mapcolor({a},{b})'

        if 0 < len(self.state) <= 10 and len(self.state[0]) <= 10:
            yield 'pattern_repeat'

            for x in range(len(self.state)):
                for y in range(len(self.state[0])):
                    for target_color in range(10):
                        yield f"flood_fill({x},{y},{target_color})"

            for x1 in range(len(self.state)):
                for y1 in range(len(self.state[0])):
                    for x2 in range(x1, len(self.state)):
                        for y2 in range(y1, len(self.state[0])):
                            yield f"crop({x1},{y1},{x2},{y2})"

        for factor in [0.5, 2]:
            yield f"scale({factor})"

        for degrees in [90, 180, 270]:
            yield f"rotate({degrees})"


class ARCPuzzleProblem(Problem):
    """
    Py-Search Problem Class based on Arc Puzzle
    """
    visited = set()

    def min_cost_heuristic(self, node):
        """
        Computes a heuristic cost based on mismatch scores, considering
        transformations (like scaling, rotation, and mirroring) that could
        minimize differences between the current and goal states. This heuristic
        aims to account for color and dimensional differences and any necessary
        transformations.
        """
        self.visited.add(node)
        current_state = node.state.state
        goal_state = self.goal.state.state

        if len(current_state) == 0:
            return len(goal_state) * len(goal_state[0])

        def mismatch_score(grid1, grid2):
            """
            Calculates the number of mismatched cells between two grids.
            """
            return sum(
                1 for r1, r2 in zip(grid1, grid2) for c1, c2 in zip(r1, r2) if c1 != c2
            )

        # def resize_if_needed(grid, target_height, target_width):
        #     """
        #     Resizes the grid to the specified target dimensions.
        #     """
        #     return node.state.resize(grid, target_height, target_width)

        # def transform_and_evaluate():
        #     """
        #     Attempts various transformations (scaling, rotation, and mirroring)
        #     on the current state to find the lowest mismatch score with the goal state.
        #     """
        #     current_height, current_width = len(
        #         current_state), len(current_state[0])
        #     goal_height, goal_width = len(goal_state), len(goal_state[0])
        #     transformations = []

        #     # Resize current state to match goal dimensions (if different)
        #     resized_current = resize_if_needed(
        #         current_state, goal_height, goal_width)
        #     transformations.append(mismatch_score(resized_current, goal_state))

        #     # Test mirroring transformations
        #     mirrored_horizontal = node.state.hmirror(resized_current)
        #     mirrored_vertical = node.state.vmirror(resized_current)
        #     transformations.extend([
        #         mismatch_score(mirrored_horizontal, goal_state),
        #         mismatch_score(mirrored_vertical, goal_state)
        #     ])

        #     # Test rotation transformations
        #     for degrees in [90, 180, 270]:
        #         rotated = node.state.rotate(resized_current, degrees)
        #         rotated_resized = resize_if_needed(
        #             rotated, goal_height, goal_width)
        #         transformations.append(
        #             mismatch_score(rotated_resized, goal_state))

        #     return min(transformations)
        base_mismatch = mismatch_score(current_state, goal_state)

        leftover_cost = abs(len(current_state) - len(goal_state)) * len(goal_state[0]) + abs(
            len(current_state[0]) - len(goal_state[0])
        ) * len(goal_state)

        final_cost = base_mismatch + leftover_cost

        return final_cost

    def random_node(self):
        return self.initial

    def node_value(self, node):
        """
        The value of a node is the combination of the node cost and the
        min_cost heuristic
        """
        return node.cost() + self.min_cost_heuristic(node)

    def successors(self, node):
        """
        An iterator that yields the successors of the provided node.
        """
        current_puzzle = node.state
        cost = node.cost()

        for action in current_puzzle.legalActions():
            next_puzzle = current_puzzle.copy()
            next_puzzle.executeAction(action)
            if next_puzzle not in self.visited:
                yield Node(next_puzzle, node, action, cost + 1)


def arc_planning(input_grid, output_grid):
    problem = ARCPuzzleProblem(initial=ARCPuzzle(
        input_grid), goal=ARCPuzzle(output_grid))

    # for solution in breadth_first_search(problem, depth_limit=2):
    #     return list(solution.path())
    for solution in local_beam_search(problem, beam_width=5, max_sideways=5):
        return list(solution.path())
    return []


def execute_actions(input_grid, actions):
    input_problem = ARCPuzzle(input_grid)
    for action in actions:
        input_problem.executeAction(action)
    return input_problem.get_matrix()

In [14]:
import json
from tqdm import tqdm
from collections import Counter

submission = {}

for key, task in tqdm(evaluation_challenges.items()):

    train_inputs = [item['input'] for item in task['train']]
    train_outputs = [item['output'] for item in task['train']]

    submission[key] = []

    all_actions = []
    # print('hi')
    for input_grid, output_grid in zip(train_inputs, train_outputs):
        # print(input_grid)
        # print(output_grid)
        actions = tuple(arc_planning(input_grid, output_grid))
        all_actions.append(actions)

    if all_actions == []:
        blank_prediction = [[0, 0], [0, 0]]
        submission[key] = [{'attempt_1': blank_prediction,
                            'attempt_2': blank_prediction} for item in task['test']]
        continue

    action_counts = Counter(all_actions)
    most_common_actions = action_counts.most_common(2)

    for test_item in task['test']:
        input_grid = test_item['input']

        attempt_1 = execute_actions(
            input_grid, list(most_common_actions[0][0]))
        if len(most_common_actions) > 1:
            attempt_2 = execute_actions(
                input_grid, list(most_common_actions[1][0]))
        else:
            attempt_2 = attempt_1.copy()

        submission[key].append({
            'attempt_1': attempt_1,
            'attempt_2': attempt_2
        })

        # print(submission[key])


# Write the submissions to file, so that they will get evaluated
with open('submission.json', 'w') as fp:
    json.dump(submission, fp)

100%|██████████| 400/400 [03:26<00:00,  1.94it/s]


# Scoring Your Submission

If you do not want to wait for gradescope to score your solution, we have provided the following code to score your submission. Note that the maximum possibe score is 400.

In [15]:
def score_submission():
    with open('../input/arc-prize-2024/arc-agi_evaluation_solutions.json', 'r') as sol_file:
        solutions = json.load(sol_file)

    with open('submission.json', 'r') as sub_file:
        submission = json.load(sub_file)

    overall_score = 0

    for task in solutions:
        score = 0
        for i, answer in enumerate(solutions[task]):
            attempt1_correct = submission[task][i]['attempt_1'] == answer
            attempt2_correct = submission[task][i]['attempt_2'] == answer
            score += int(attempt1_correct or attempt2_correct)

        score /= len(solutions[task])

        overall_score += score

    print(overall_score)

You can run the above code by uncommenting the following code block.

In [16]:
score_submission()

7.5


# Confused about where to get started?

If you're not sure what an initial solution might look like, then consider looking at public notebooks here: https://www.kaggle.com/competitions/arc-prize-2024/code or joining the public discussion here: https://www.kaggle.com/competitions/arc-prize-2024/discussion.

One example notebook that uses a very simple knowledge-based approach is this one: https://www.kaggle.com/code/michaelhodel/program-synthesis-starter-notebook/notebook, which conducts search over a space of domain specific block languages to form hypotheses and then applies these to test items.